In [9]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("Chưa bật GPU trong Kaggle Settings!")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM (GB): 14.56


In [10]:
!pip install -q ultralytics pyyaml

In [11]:
from pathlib import Path

for p in Path("/kaggle/input").iterdir():
    print(p)

/kaggle/input/datasets


In [12]:
from pathlib import Path
import yaml

input_root = Path("/kaggle/input")

yaml_files = list(input_root.rglob("data.yaml"))
model_files = list(input_root.rglob("ppe_best.pt"))

assert len(yaml_files) == 1, f"data.yaml: {yaml_files}"
assert len(model_files) == 1, f"ppe_best.pt: {model_files}"

data_yaml = yaml_files[0]
data_root = data_yaml.parent
checkpoint = model_files[0]

print("Dataset:", data_root)
print("YAML:", data_yaml)
print("Model:", checkpoint)

Dataset: /kaggle/input/datasets/duyhoang17930/ppe-enriched-v1/ppe_enriched
YAML: /kaggle/input/datasets/duyhoang17930/ppe-enriched-v1/ppe_enriched/data.yaml
Model: /kaggle/input/datasets/duyhoang17930/ppe-start-checkpoint/ppe_best.pt


In [13]:
cfg = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))

cfg["path"] = str(data_root)
cfg["train"] = "images/train"
cfg["val"] = "images/val"
cfg["test"] = "images/test"

train_yaml = Path("/kaggle/working/ppe_enriched.yaml")

train_yaml.write_text(
    yaml.safe_dump(cfg, sort_keys=False),
    encoding="utf-8"
)

print(train_yaml.read_text())

path: /kaggle/input/datasets/duyhoang17930/ppe-enriched-v1/ppe_enriched
train: images/train
val: images/val
test: images/test
names:
  0: person
  1: helmet
  2: vest



In [14]:
extensions = {".jpg", ".jpeg", ".png", ".webp"}

for split in ("train", "val", "test"):
    image_dir = data_root / "images" / split
    label_dir = data_root / "labels" / split

    assert image_dir.is_dir(), f"Missing {image_dir}"
    assert label_dir.is_dir(), f"Missing {label_dir}"

    images = [
        p for p in image_dir.iterdir()
        if p.suffix.lower() in extensions
    ]
    labels = list(label_dir.glob("*.txt"))

    print(f"{split}: {len(images)} images, {len(labels)} labels")

print("Classes:", cfg["names"])

train: 8396 images, 8396 labels
val: 275 images, 275 labels
test: 171 images, 171 labels
Classes: {0: 'person', 1: 'helmet', 2: 'vest'}


In [16]:
from ultralytics import YOLO
from pathlib import Path

run_dir = Path("/kaggle/working/ppe_runs")

model = YOLO(str(checkpoint))

results = model.train(
    data=str(train_yaml),
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project=str(run_dir),
    name="ppe_enriched",
    patience=20,
    exist_ok=True,
    plots=True
)

Ultralytics 8.4.197 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/ppe_enriched.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/datasets/duyhoang17930/ppe-start-checkpoint/ppe_best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, na

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/60      3.01G      1.127     0.9331      1.404        121        640: 100% ━━━━━━━━━━━━ 525/525 7.5it/s 1:100.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.8it/s 1.9s0.2s
                   all        275       1577      0.832      0.772      0.818      0.439

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/60      3.01G      1.144     0.9614      1.411        104        640: 100% ━━━━━━━━━━━━ 525/525 7.6it/s 1:090.1s4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 5.2it/s 1.7s0.2s
                   all        275       1577      0.856      0.746      0.821      0.448

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/60      3.01G       1.15     0.9535      1.406        124        640: 100% ━━━━━━━━━━━━ 525/525 7.6it/s 1:090.1sss
                 Class     Images 

In [17]:
best_path = run_dir / "ppe_enriched" / "weights" / "best.pt"

assert best_path.exists(), f"Không tìm thấy: {best_path}"

best = YOLO(str(best_path))

metrics = best.val(
    data=str(train_yaml),
    split="test",
    imgsz=640,
    device=0,
    plots=True
)

print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)
print("Precision:", metrics.box.mp)
print("Recall:", metrics.box.mr)

print("Classes:", best.names)

for class_id, ap in zip(
    metrics.box.ap_class_index,
    metrics.box.ap50
):
    print(best.names[int(class_id)], "AP50:", round(float(ap), 4))

Ultralytics 8.4.197 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.4±0.1 ms, read: 0.5±0.2 MB/s, size: 87.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /kaggle/input/datasets/duyhoang17930/ppe-enriched-v1/ppe_enriched/labels/test... 171 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 171/171 9.0it/s 18.9s<0.2s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/duyhoang17930/ppe-enriched-v1/ppe_enriched/labels is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 2.5s/it 27.8s0.3s
                   all        171        926      0.902      0.824       0.88      0.523
                person        171        371       0.89      

In [18]:
onnx_path = best.export(
    format="onnx",
    imgsz=320
)

print("ONNX:", onnx_path)

Ultralytics 8.4.197 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO11n summary (fused): 100 layers, 2,582,737 parameters, 0 gradients, 1.6 GFLOPs

PyTorch: starting from '/kaggle/working/ppe_runs/ppe_enriched/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 7, 2100) (5.2 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 12 packages in 304ms
Prepared 2 packages in 298ms
Installed 2 packages in 12ms
 + onnxruntime==1.31.1
 + onnxslim==0.1.98

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.98...
ONNX: export success ✅ 3.0

In [19]:
import shutil
from pathlib import Path

output = Path("/kaggle/working/ppe_result")

output.mkdir(exist_ok=True)

shutil.copy2(best_path, output / "ppe_best_enriched.pt")

if Path(onnx_path).exists():
    shutil.copy2(onnx_path, output / "ppe_best_enriched.onnx")

shutil.make_archive(
    "/kaggle/working/ppe_result",
    "zip",
    output
)

print("Created: /kaggle/working/ppe_result.zip")

Created: /kaggle/working/ppe_result.zip
